# Fine-tune DINOv2-Large + UperNet (Full Mapping, Unfrozen Backbone)

Thin orchestration notebook for Databricks. All reusable logic lives in
`histological_image_analysis.training` (installed as a wheel on the cluster).

**Full mapping with partial backbone fine-tuning:** Unfreeze the last 4 DINOv2
transformer blocks (20-23) to adapt pretrained features to the Nissl histology
domain. Uses differential learning rate: 1e-5 for backbone, 1e-4 for UperNet head.

**Why:** DINOv2 was pretrained on natural images. Nissl-stained brain tissue has
fundamentally different visual characteristics — cell density patterns, laminar
organization, and staining gradients. Frozen features cannot capture these.

**Memory optimizations (required — batch=4 OOM'd on L40S 48 GB):**
- `per_device_train_batch_size=1` + `gradient_accumulation_steps=4` (effective batch=4)
- Gradient checkpointing on backbone only (`model.backbone.gradient_checkpointing_enable()`)
  - NOTE: UperNet as a whole does NOT support gradient checkpointing (ValueError).
    The backbone (DINOv2) does. Call it on the backbone directly.
- `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` to reduce fragmentation
- `torch.cuda.empty_cache()` after sanity-check forward pass

**Cluster:** Single GPU (1x L40S 48 GB). Do NOT use multi-GPU — DDP replicates
the full model per GPU and adds overhead (see docs/step10_gpu_memory_review.md).

In [0]:
# Cell 0 — Install project wheel from DBFS
# The wheel is uploaded by `make deploy-wheel` and contains all reusable
# training pipeline code (ontology, slicer, dataset, training modules).

%pip install /dbfs/FileStore/wheels/histological_image_analysis-0.1.0-py3-none-any.whl
dbutils.library.restartPython()

Processing /dbfs/FileStore/wheels/histological_image_analysis-0.1.0-py3-none-any.whl
histological-image-analysis is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Cell 1 — Configuration
# All environment-specific paths and hyperparameters live here.

import os
import mlflow

# ---------- CUDA memory optimization ----------
# Reduce fragmentation — expandable_segments avoids "reserved but unallocated" waste
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ---------- MLflow setup ----------
os.environ["MLFLOW_ENABLE_SYSTEM_METRICS_LOGGING"] = "true"

EXPERIMENT_NAME = "/Users/noel.nosse@grainger.com/histology-brain-segmentation"
try:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
except Exception:
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
mlflow.set_experiment(experiment_id=experiment_id)
print(f"MLflow experiment: {EXPERIMENT_NAME} (ID: {experiment_id})")

# ---------- Databricks paths ----------
WORKSPACE_BASE = "/Workspace/Users/noel.nosse@grainger.com/visual-model-ft/histology"
ONTOLOGY_PATH = f"{WORKSPACE_BASE}/ontology/structure_graph_1.json"
ANNOTATION_10_PATH = f"{WORKSPACE_BASE}/ccfv3/annotation_10.nrrd"
# ara_nissl_10.nrrd exceeds workspace limit — lives on DBFS
NISSL_10_PATH = "/dbfs/FileStore/allen_brain_data/ccfv3/ara_nissl_10.nrrd"

# ---------- JFrog / HuggingFace model ----------
HF_ENDPOINT = os.environ.get(
    "HF_ENDPOINT",
    "https://graingerreadonly.jfrog.io/artifactory/api/huggingfaceml/huggingfaceml-remote",
)
MODEL_REPO_ID = "facebook/dinov2-large"
MODEL_CACHE_DIR = "/tmp/dinov2-large"

# ---------- Mapping ----------
MAPPING_TYPE = "full"

# ---------- Training hyperparameters ----------
# Key change from finetune_full.ipynb: backbone is partially unfrozen
# with differential learning rate (10x lower for backbone).
#
# Memory fix: batch=4 OOM'd on L40S 48 GB. Use batch=1 + grad_accum=4
# for the same effective batch size with 1/4 peak memory.
BACKBONE_LR = 1e-5
HEAD_LR = 1e-4
NUM_UNFROZEN_BLOCKS = 4  # Unfreeze last N transformer blocks (20-23)
BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 4  # Effective batch = BATCH_SIZE * GRAD_ACCUM = 4

HYPERPARAMS = {
    "mapping_type": MAPPING_TYPE,
    "crop_size": 518,  # DINOv2 native resolution
    "batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "effective_batch_size": BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
    "backbone_lr": BACKBONE_LR,
    "head_lr": HEAD_LR,
    "num_unfrozen_blocks": NUM_UNFROZEN_BLOCKS,
    "num_epochs": 100,  # Doubled from 50 — early stopping will halt if converged
    "freeze_backbone": False,  # Partially unfrozen (see Cell 4)
    "gradient_checkpointing": "backbone_only",
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "split_strategy": "interleaved",
    "model": MODEL_REPO_ID,
    "dataset": "CCFv3 ara_nissl_10",
}

# Unpack for convenience (NUM_LABELS computed dynamically in Cell 3)
CROP_SIZE = HYPERPARAMS["crop_size"]
NUM_EPOCHS = HYPERPARAMS["num_epochs"]
SPLIT_STRATEGY = HYPERPARAMS["split_strategy"]

# ---------- Output ----------
OUTPUT_DIR = "/tmp/checkpoints/unfrozen"
FINAL_MODEL_DIR = "/dbfs/FileStore/allen_brain_data/models/unfrozen"

print(f"Ontology:           {ONTOLOGY_PATH}")
print(f"Nissl 10\u00b5m:         {NISSL_10_PATH}")
print(f"HF endpoint:        {HF_ENDPOINT}")
print(f"Mapping:            {MAPPING_TYPE}")
print(f"Batch size:         {BATCH_SIZE} (x{GRADIENT_ACCUMULATION_STEPS} grad accum = effective {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS})")
print(f"Backbone LR:        {BACKBONE_LR}")
print(f"Head LR:            {HEAD_LR}")
print(f"Unfrozen blocks:    last {NUM_UNFROZEN_BLOCKS} (blocks {24 - NUM_UNFROZEN_BLOCKS}-23)")
print(f"Grad checkpointing: backbone only (UperNet does not support it)")
print(f"CUDA alloc config:  expandable_segments:True")
print(f"Epochs:             {NUM_EPOCHS} (with early stopping)")
print(f"Split strategy:     {SPLIT_STRATEGY}")
print(f"Checkpoints:        {OUTPUT_DIR}")
print(f"Final model:        {FINAL_MODEL_DIR}")

MLflow experiment: /Users/noel.nosse@grainger.com/histology-brain-segmentation (ID: 1345391216675532)
Ontology:           /Workspace/Users/noel.nosse@grainger.com/visual-model-ft/histology/ontology/structure_graph_1.json
Nissl 10µm:         /dbfs/FileStore/allen_brain_data/ccfv3/ara_nissl_10.nrrd
HF endpoint:        https://graingerreadonly.jfrog.io/artifactory/api/huggingfaceml/huggingfaceml-remote
Mapping:            full
Batch size:         1 (x4 grad accum = effective 4)
Backbone LR:        1e-05
Head LR:            0.0001
Unfrozen blocks:    last 4 (blocks 20-23)
Grad checkpointing: backbone only (UperNet does not support it)
CUDA alloc config:  expandable_segments:True
Epochs:             100 (with early stopping)
Split strategy:     interleaved
Checkpoints:        /tmp/checkpoints/unfrozen
Final model:        /dbfs/FileStore/allen_brain_data/models/unfrozen


In [0]:
# Cell 2 — Download model weights from JFrog Artifactory mirror
#
# Uses retry pattern per LESSONS_LEARNED.md — Artifactory can drop connections.
# etag_timeout=86400 avoids unnecessary HEAD requests.

import time
from huggingface_hub import snapshot_download

print(f"Downloading {MODEL_REPO_ID} from Artifactory...")
for attempt in range(1, 4):
    try:
        model_path = snapshot_download(
            repo_id=MODEL_REPO_ID,
            endpoint=HF_ENDPOINT,
            etag_timeout=86400,
            cache_dir=MODEL_CACHE_DIR,
        )
        break
    except Exception as e:
        if attempt < 3:
            wait = 2 ** attempt
            print(f"  Attempt {attempt} failed ({e.__class__.__name__}), retrying in {wait}s...")
            time.sleep(wait)
        else:
            raise

print(f"Model downloaded to: {model_path}")

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Model downloaded to: /tmp/dinov2-large/models--facebook--dinov2-large/snapshots/a18992645e496e6d21ea90829a6652ed6f75ec47


In [0]:
# Cell 3 — Build training and validation datasets
#
# Pipeline: OntologyMapper -> CCFv3Slicer -> BrainSegmentationDataset
#
# Uses full mapping (every structure -> unique class) and INTERLEAVED split.

from histological_image_analysis.ontology import OntologyMapper
from histological_image_analysis.ccfv3_slicer import CCFv3Slicer
from histological_image_analysis.dataset import BrainSegmentationDataset

# 1. Load ontology and build full mapping
mapper = OntologyMapper(ONTOLOGY_PATH)
mapping = mapper.build_full_mapping()
NUM_LABELS = mapper.get_num_labels(mapping)
class_names = mapper.get_class_names(mapping)
HYPERPARAMS["num_labels"] = NUM_LABELS
print(f"Full mapping: {NUM_LABELS} classes (including background)")

# 2. Load CCFv3 volumes (10um Nissl + annotations)
slicer = CCFv3Slicer(
    image_path=NISSL_10_PATH,
    annotation_path=ANNOTATION_10_PATH,
    ontology_mapper=mapper,
)
slicer.load_volumes()
print(f"Loaded {slicer.num_slices} coronal slices")

# 3. Interleaved train/val/test split
splits = slicer.get_split_indices(
    train_frac=0.8, val_frac=0.1, split_strategy=SPLIT_STRATEGY,
)
print(f"Train: {len(splits['train'])} | Val: {len(splits['val'])} | Test: {len(splits['test'])}")

# 4. Create datasets
train_ds = BrainSegmentationDataset(
    slicer=slicer, split="train", mapping=mapping,
    crop_size=CROP_SIZE, augment=True, split_strategy=SPLIT_STRATEGY,
)
val_ds = BrainSegmentationDataset(
    slicer=slicer, split="val", mapping=mapping,
    crop_size=CROP_SIZE, augment=False, split_strategy=SPLIT_STRATEGY,
)
print(f"Train samples: {len(train_ds)} | Val samples: {len(val_ds)}")

# Sanity check: inspect one sample
sample = train_ds[0]
print(f"pixel_values: {sample['pixel_values'].shape}, labels: {sample['labels'].shape}")
print(f"Unique classes in sample: {len(set(sample['labels'].numpy().ravel()))}")

Full mapping: 1328 classes (including background)
Loaded 1320 coronal slices
Train: 1016 | Val: 127 | Test: 127
Train samples: 1016 | Val samples: 127
pixel_values: torch.Size([3, 518, 518]), labels: torch.Size([518, 518])
Unique classes in sample: 4


In [0]:
# Cell 4 — Create model with partially unfrozen backbone
#
# Strategy: Unfreeze last 4 transformer blocks (20-23) + layernorm.
# Keep blocks 0-19 and embeddings frozen to preserve low-level features
# and minimize overfitting risk on 1,016 training samples.
#
# Memory optimizations:
# - Gradient checkpointing on BACKBONE ONLY. UperNet as a whole raises
#   ValueError if you pass gradient_checkpointing=True to TrainingArguments.
#   But the DINOv2 backbone supports it. Call model.backbone.gradient_checkpointing_enable().
# - torch.cuda.empty_cache() after sanity-check forward pass

import torch
from histological_image_analysis.training import create_model

# Create model with ALL backbone parameters unfrozen
model = create_model(
    num_labels=NUM_LABELS,
    freeze_backbone=False,  # Start fully unfrozen, then selectively freeze below
    pretrained_backbone_path=model_path,
)

# Selectively freeze backbone: keep only last N blocks + layernorm trainable
first_frozen_block = 24 - NUM_UNFROZEN_BLOCKS  # block 20 for N=4

# Freeze embeddings
for param in model.backbone.embeddings.parameters():
    param.requires_grad = False

# Freeze encoder blocks 0 through (first_frozen_block - 1)
for i in range(first_frozen_block):
    for param in model.backbone.encoder.layer[i].parameters():
        param.requires_grad = False

# Enable gradient checkpointing on BACKBONE ONLY
# This trades recomputation for ~40-60% activation memory savings.
# UperNet does NOT support it (ValueError), but DINOv2 backbone does.
model.backbone.gradient_checkpointing_enable()
print("Gradient checkpointing: ENABLED (backbone only)")

# Verify: count trainable params by component
backbone_frozen = sum(p.numel() for p in model.backbone.parameters() if not p.requires_grad)
backbone_trainable = sum(p.numel() for p in model.backbone.parameters() if p.requires_grad)
head_trainable = sum(
    p.numel() for n, p in model.named_parameters()
    if p.requires_grad and "backbone" not in n
)
total = sum(p.numel() for p in model.parameters())

print(f"Backbone frozen:    {backbone_frozen:>12,} params")
print(f"Backbone trainable: {backbone_trainable:>12,} params (blocks {first_frozen_block}-23 + layernorm)")
print(f"Head trainable:     {head_trainable:>12,} params (decode_head + auxiliary_head)")
print(f"Total trainable:    {backbone_trainable + head_trainable:>12,} / {total:,} ({(backbone_trainable + head_trainable) / total * 100:.1f}%)")

# Sanity check — single forward pass
model.eval()
with torch.no_grad():
    dummy = torch.randn(1, 3, CROP_SIZE, CROP_SIZE)
    if torch.cuda.is_available():
        model = model.cuda()
        dummy = dummy.cuda()
    out = model(pixel_values=dummy)
    print(f"\nLogits shape: {out.logits.shape}")
    print(f"Expected: (1, {NUM_LABELS}, {CROP_SIZE}, {CROP_SIZE})")
    logits_mb = out.logits.element_size() * out.logits.nelement() / 1e6
    print(f"Logits memory (single sample): {logits_mb:.1f} MB")
    assert out.logits.shape == (1, NUM_LABELS, CROP_SIZE, CROP_SIZE)

# Free GPU memory from sanity check (per LESSONS_LEARNED.md)
del out, dummy
torch.cuda.empty_cache()
model = model.cpu()
torch.cuda.empty_cache()

# Print GPU status
import subprocess
print("\nGPU memory after cleanup:")
print(subprocess.check_output(["nvidia-smi", "--query-gpu=memory.used,memory.free,memory.total",
                                "--format=csv,noheader,nounits"]).decode().strip(), "MiB (used, free, total)")
print("Forward pass OK")

2026-03-11 17:06:21.969984: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-11 17:06:21.985792: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-11 17:06:21.990620: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-11 17:06:22.003666: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-11 17:06:23.101220: W tensorflow/compiler/tf2

Gradient checkpointing: ENABLED (backbone only)
Backbone frozen:     253,973,504 params
Backbone trainable:   50,395,136 params (blocks 20-23 + layernorm)
Head trainable:       37,735,520 params (decode_head + auxiliary_head)
Total trainable:      88,130,656 / 342,104,160 (25.8%)

Logits shape: torch.Size([1, 1328, 518, 518])
Expected: (1, 1328, 518, 518)
Logits memory (single sample): 1425.3 MB

GPU memory after cleanup:
530, 45065, 46068 MiB (used, free, total)
Forward pass OK


In [0]:
# Cell 5 — Train with differential learning rate
#
# Single MLflow run spans training + eval + export (per LESSONS_LEARNED.md).
#
# Key differences from finetune_full.ipynb:
# - Custom optimizer with two param groups (backbone 1e-5, head 1e-4)
# - batch_size=1 + gradient_accumulation_steps=4 (effective batch=4, peak memory quartered)
# - ddp_find_unused_parameters=False (all params are used with unfrozen backbone)

import torch
from datetime import datetime
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from histological_image_analysis.training import (
    get_training_args,
    make_compute_metrics,
    preprocess_logits_for_metrics,
)
from transformers import Trainer

mlflow.start_run(run_name=f"unfrozen-{NUM_LABELS}class-{datetime.now().strftime('%Y%m%d-%H%M')}")
mlflow.log_params(HYPERPARAMS)

# Build training args
training_args = get_training_args(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=HEAD_LR,  # Shown in logs; actual LR controlled by optimizer below
    fp16=torch.cuda.is_available(),
    report_to="mlflow",
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    ddp_find_unused_parameters=False,  # All params used now that backbone is unfrozen
)

# Build optimizer with differential learning rate
backbone_params = [
    p for n, p in model.named_parameters()
    if p.requires_grad and "backbone" in n
]
head_params = [
    p for n, p in model.named_parameters()
    if p.requires_grad and "backbone" not in n
]

optimizer = AdamW(
    [
        {"params": backbone_params, "lr": BACKBONE_LR},
        {"params": head_params, "lr": HEAD_LR},
    ],
    weight_decay=HYPERPARAMS["weight_decay"],
)

# Build linear warmup scheduler
# Account for gradient accumulation: steps = samples / (batch * accum)
num_training_steps = (len(train_ds) // (BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)) * NUM_EPOCHS
num_warmup_steps = int(num_training_steps * HYPERPARAMS["warmup_ratio"])

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

print(f"Optimizer: AdamW with 2 param groups")
print(f"  Backbone ({len(backbone_params)} tensors): lr={BACKBONE_LR}")
print(f"  Head ({len(head_params)} tensors):     lr={HEAD_LR}")
print(f"  Batch size: {BATCH_SIZE} x {GRADIENT_ACCUMULATION_STEPS} grad accum = effective {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"  Total training steps: {num_training_steps}")
print(f"  Warmup steps: {num_warmup_steps}")

# Build trainer with custom optimizer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=make_compute_metrics(NUM_LABELS),
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
    optimizers=(optimizer, scheduler),
)

trainer.train()

2026/03/11 17:06:34 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/03/11 17:06:34 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.


Optimizer: AdamW with 2 param groups
  Backbone (74 tensors): lr=1e-05
  Head (43 tensors):     lr=0.0001
  Batch size: 1 x 4 grad accum = effective 4
  Total training steps: 25400
  Warmup steps: 2540
[2026-03-11 17:06:35,046] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -lcufile: No such file or directory
collect2: error: ld returned 1 exit status
Trainer is attempting to log a value of "{0: 'LABEL_0', 1: 'LABEL_1', 2: 'LABEL_2', 3: 'LABEL_3', 4: 'LABEL_4', 5: 'LABEL_5', 6: 'LABEL_6', 7: 'LABEL_7', 8: 'LABEL_8', 9: 'LABEL_9', 10: 'LABEL_10', 11: 'LABEL_11', 12: 'LABEL_12', 13: 'LABEL_13', 14: 'LABEL_14', 15: 'LABEL_15', 16: 'LABEL_16', 17: 'LABEL_17', 18: 'LABEL_18', 19: 'LABEL_19', 20: 'LABEL_20', 21: 'LABEL_21', 22: 'LABEL_22', 23: 'LABEL_23', 24: 'LABEL_24', 25: 'LABEL_25', 26: 'LABEL_26', 27: 'LABEL_27', 28: 'LABEL_28', 29: 'LABEL_29', 30: 'LABEL_30', 31: 'LABEL_31', 32: 'LABEL_32', 33: 'LABEL_33', 34: 'LABEL_34', 35: 'LABEL_35', 36: 'LABEL_36', 37: 'LABEL_37', 38: 'LABEL_38', 39: 'LABEL_39', 40: 'LABEL_40', 41: 'LABEL_41', 42: 'LABEL_42', 43: 'LABEL_43', 44: 'LABEL_44', 45: 'LABEL_45', 46: 'LABEL_46', 47: 'LABEL_47', 48: '

---------------------------------------------------------------------------
ValueError                                Traceback (most recent call last)
File <command-8083493659665641>, line 84
     73 # Build trainer with custom optimizer
     74 trainer = Trainer(
     75     model=model,
     76     args=training_args,
   (...)
     81     optimizers=(optimizer, scheduler),
     82 )
---> 84 trainer.train()

File /databricks/python/lib/python3.12/site-packages/mlflow/utils/autologging_utils/safety.py:402, in safe_patch.<locals>.safe_patch_function(*args, **kwargs)
    384 if (
    385     active_session_failed
    386     or autologging_is_disabled(autologging_integration)
   (...)
    396     # warning behavior during original function execution, since autologging is being
    397     # skipped
    398     with NonMlflowWarningsBehaviorForCurrentThread(
    399         disable_warnings=False,
    400         reroute_warnings=False,
    401     ):
--> 402         return original(*arg

In [0]:
# Cell 6 — Evaluate + Visualize
#
# Logs to the same active MLflow run opened in Cell 5.
# Uses get_class_names() for per-class IoU reporting.

import matplotlib.pyplot as plt
import numpy as np

# Run evaluation
eval_results = trainer.evaluate()
print("Evaluation results:")
for k, v in sorted(eval_results.items()):
    if isinstance(v, float) and not k.startswith("eval_iou_class_"):
        print(f"  {k}: {v:.4f}")

# Log final eval metrics explicitly
mlflow.log_metrics({
    "final_mean_iou": eval_results.get("eval_mean_iou", 0.0),
    "final_overall_accuracy": eval_results.get("eval_overall_accuracy", 0.0),
})

# Per-class IoU summary — show top and bottom classes
class_ious = {}
for cls in range(NUM_LABELS):
    iou_key = f"eval_iou_class_{cls}"
    iou = eval_results.get(iou_key, float('nan'))
    if not np.isnan(iou):
        class_ious[cls] = iou

sorted_ious = sorted(class_ious.items(), key=lambda x: x[1], reverse=True)
print(f"\nTop 10 classes by IoU:")
for cls, iou in sorted_ious[:10]:
    name = class_names[cls] if cls < len(class_names) else f"class_{cls}"
    print(f"  {cls:4d} ({name:40s}): {iou:.4f}")

print(f"\nBottom 10 classes by IoU:")
for cls, iou in sorted_ious[-10:]:
    name = class_names[cls] if cls < len(class_names) else f"class_{cls}"
    print(f"  {cls:4d} ({name:40s}): {iou:.4f}")

print(f"\nClasses with non-NaN IoU: {len(class_ious)} / {NUM_LABELS}")

# Compare with frozen backbone baseline (Run 4: 60.3% mIoU)
frozen_miou = 0.603
current_miou = eval_results.get('eval_mean_iou', 0.0)
delta = current_miou - frozen_miou
print(f"\n--- Comparison with frozen backbone (Run 4) ---")
print(f"Frozen mIoU:   {frozen_miou:.1%}")
print(f"Unfrozen mIoU: {current_miou:.1%}")
print(f"Delta:         {delta:+.1%}")

# Visualize predictions vs ground truth on a few val samples
model.eval()
fig, axes = plt.subplots(3, 3, figsize=(15, 15))

for i in range(3):
    sample = val_ds[i * (len(val_ds) // 3)]
    pixel_values = sample["pixel_values"].unsqueeze(0)
    labels = sample["labels"].numpy()

    with torch.no_grad():
        if torch.cuda.is_available():
            pixel_values = pixel_values.cuda()
        logits = model(pixel_values=pixel_values).logits
        preds = logits.argmax(dim=1).squeeze().cpu().numpy()

    # Denormalize image for display
    img = sample["pixel_values"].permute(1, 2, 0).numpy()
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)

    axes[i, 0].imshow(img)
    axes[i, 0].set_title("Input")
    axes[i, 1].imshow(labels, cmap="nipy_spectral")
    axes[i, 1].set_title("Ground Truth")
    axes[i, 2].imshow(preds, cmap="nipy_spectral")
    axes[i, 2].set_title("Prediction")

for ax in axes.ravel():
    ax.axis("off")

plt.suptitle(
    f"Unfrozen backbone | {MAPPING_TYPE} {NUM_LABELS}-class | "
    f"mIoU={current_miou:.3f} (delta={delta:+.3f} vs frozen)"
)
plt.tight_layout()
plt.show()

In [0]:
# Cell 7 — Save final model + close MLflow run

import os

os.makedirs(FINAL_MODEL_DIR, exist_ok=True)
trainer.save_model(FINAL_MODEL_DIR)
print(f"Model saved to: {FINAL_MODEL_DIR}")

# Close the MLflow run opened in Cell 5.
# Per LESSONS_LEARNED.md: HF Trainer's MLflowCallback can leave runs open.
mlflow.end_run()
print("MLflow run closed.")